# Pre-process Amazon Reviews 2023

Same steps as `process.ipynb`, using `package.preprocess` / `package.utils` instead of in-notebook defs.

# 0. Import & logging

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "package").is_dir())
sys.path.insert(0, str(ROOT))

from package.utils.data import amazon_processed_dir, amazon_raw_dir
from package.utils.log import AMAZON_PROCESS_LOG_DIR, setup_logging
from package.preprocess import (
    assign_idx,
    keep_first_filter,
    kcore_filter,
    leave_one_out_split,
    load_reviews_and_metadata,
    low_rating_filter,
    write_id_maps,
    write_products,
    write_simulator_jsonl,
    write_splits,
)

In [ ]:
logger = setup_logging(name="process", log_dir=AMAZON_PROCESS_LOG_DIR)

# 1. Configuration

In [ ]:
SEED = 2024
CATEGORY = "All_Beauty"

RATING_THRESHOLD = 3.0
USER_K = 5
ITEM_K = 5

MAX_HISTORY_LEN = 10
MAX_TITLE_LEN = 50
MAX_DESCRIPTION_SENTENCES = 2
SIMULATOR_SAMPLE_N = 900

DATA_DIR = amazon_raw_dir(CATEGORY)
OUTPUT_DIR = amazon_processed_dir(CATEGORY) / "chatbot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"raw: {DATA_DIR}")
print(f"out: {OUTPUT_DIR}")
print(f"log: {AMAZON_PROCESS_LOG_DIR}")

# 2–3. Load, clean metadata, select columns

`load_reviews_and_metadata` already does the old cells 10–17: jsonl cache, drop missing titles, join description, take first category, rename `parent_asin` → `item_id`, keep reviews whose item is in meta.

In [ ]:
review_df, meta_df = load_reviews_and_metadata(
    CATEGORY, logger, max_description_sentences=MAX_DESCRIPTION_SENTENCES
)
print(review_df.shape, meta_df.shape)
review_df.head()

# 4. Filter

In [ ]:
data_df = keep_first_filter(review_df)
logger.info("After keep-first filter: %s", data_df.shape)

data_df = low_rating_filter(data_df, RATING_THRESHOLD)
logger.info("After rating filter: %s", data_df.shape)

data_df = kcore_filter(data_df, USER_K, ITEM_K)
logger.info("After k-core filter: %s", data_df.shape)

# 5. Map user and item ids

In [ ]:
data_df, user_map, item_map = assign_idx(data_df)
write_id_maps(item_map, user_map, OUTPUT_DIR)
logger.info("Saved id maps to %s", OUTPUT_DIR / "map.json")

# 6. Leave-one-out split

In [ ]:
df_train, df_valid, df_test, df_train_0 = leave_one_out_split(data_df)
write_splits(df_train, df_valid, df_test, df_train_0, OUTPUT_DIR)
logger.info(
    "Saved splits to %s (train=%d, valid=%d, test=%d, full_history=%d)",
    OUTPUT_DIR, len(df_train), len(df_valid), len(df_test), len(df_train_0),
)

# 7. Product table

In [ ]:
saved_meta_df = write_products(meta_df, item_map, df_train_0, OUTPUT_DIR)
logger.info("Saved product table (%d items) to %s", len(saved_meta_df), OUTPUT_DIR)

# 8. Simulator jsonl sample

In [ ]:
simulator_path = write_simulator_jsonl(
    df_test,
    df_train_0,
    saved_meta_df,
    OUTPUT_DIR,
    sample_n=SIMULATOR_SAMPLE_N,
    seed=SEED,
    max_history_len=MAX_HISTORY_LEN,
    max_title_len=MAX_TITLE_LEN,
)
logger.info("Pipeline complete: %s", simulator_path)